### Loading the Processed Dataset

The preprocessing stage produced a reduced dataset containing 12 selected sensor features together with the `class` and `state` labels.

The processed files are stored separately from the original 3W dataset, with each file/instance and its timestamp index preserved.

This notebook uses the processed dataset as the starting point for feature engineering.

In [1]:
# Import the libraries required for feature engineering.

from pathlib import Path
import pandas as pd


# Define the project and processed-data paths.

project_path = Path.cwd().parent

processed_path = (
    project_path
    / "data"
    / "processed"
    / "3W"
)


# Find all processed Parquet files.

processed_files = sorted(
    processed_path.glob("*/*.parquet")
)

print("Processed files:", len(processed_files))

Processed files: 2228


### Defining the Feature-Engineering Scope

Feature engineering is designed around the temporal nature of the 3W dataset.

Each file/instance contains time-ordered sensor observations, so the engineered features will describe recent sensor behavior rather than treating each observation independently.

The feature set will focus on:

- **Timestamp-based features** to capture time-related operating patterns.
- **Lag features** to represent recent past sensor values.
- **Rolling-window features** to summarize recent sensor behavior.
- **Rate-of-change features** to capture increasing or decreasing sensor trends.
- **Variability and trend features** to describe short-term changes in sensor behavior.

All temporal features will use information available at or before the current observation to avoid introducing future information into the model.

In [2]:
# Define the sensor columns that will be used for feature engineering.

sensor_columns = [
    "P-TPT",
    "P-MON-CKP",
    "P-PDG",
    "T-TPT",
    "T-JUS-CKP",
    "P-JUS-CKGL",
    "P-ANULAR",
    "QGL",
    "ESTADO-SDV-P",
    "T-MON-CKP",
    "P-JUS-CKP",
    "T-PDG"
]


# Define the label columns separately.

label_columns = [
    "class",
    "state"
]


print("Number of sensor features:", len(sensor_columns))
print("Number of label columns:", len(label_columns))
print("Sensor features:", sensor_columns)

Number of sensor features: 12
Number of label columns: 2
Sensor features: ['P-TPT', 'P-MON-CKP', 'P-PDG', 'T-TPT', 'T-JUS-CKP', 'P-JUS-CKGL', 'P-ANULAR', 'QGL', 'ESTADO-SDV-P', 'T-MON-CKP', 'P-JUS-CKP', 'T-PDG']


### Defining the Prediction Target Setup

The objective is to detect rare industrial events early enough to support an operational alert.

Therefore, the prediction target will be defined using future observations rather than the current `class` value alone. At each timestamp, the model will use the current and historical sensor information to determine whether a relevant event occurs within a future prediction horizon.

The original `class` and `state` columns are retained during feature engineering for target construction and validation, but they will not be used as input features.

The exact target definition and prediction horizon must be established before creating lag and rolling features so that all engineered features remain consistent with the intended alerting task.

In [4]:
# Define the columns that will not be used as model input features.

excluded_columns = [
    "class",
    "state"
]


# Confirm that the sensor and label columns are separated correctly.

feature_columns = [
    column
    for column in sensor_columns
    if column not in excluded_columns
]


print("Input sensor features:", len(feature_columns))
print("Excluded label columns:", excluded_columns)

Input sensor features: 12
Excluded label columns: ['class', 'state']


### Creating Timestamp-Based Features

The timestamp index provides temporal information that can be converted into explicit model features.

The hour of the day and day of the week are extracted to represent recurring operating patterns. These features are derived only from the timestamp and do not use future sensor or label information.

The original timestamp index is preserved for temporal ordering and later validation.

In [5]:
# Create timestamp-based features for each processed file/instance.

timestamp_features = [
    "hour",
    "day_of_week"
]


# Test the feature creation on the first processed file.

sample_file = processed_files[0]

sample_data = pd.read_parquet(sample_file)

sample_data["hour"] = sample_data.index.hour
sample_data["day_of_week"] = sample_data.index.dayofweek


print("File:", sample_file.name)
print("New timestamp features:", timestamp_features)
print()
print(sample_data[timestamp_features].head())

File: WELL-00001_20170201010207.parquet
New timestamp features: ['hour', 'day_of_week']

                     hour  day_of_week
timestamp                             
2017-02-01 01:02:07     1            2
2017-02-01 01:02:08     1            2
2017-02-01 01:02:09     1            2
2017-02-01 01:02:10     1            2
2017-02-01 01:02:11     1            2


### Meaning of the Timestamp Features

The timestamp features convert the original timestamp into numerical values that can be used by machine learning models.

- **`hour`** represents the hour of the day, from `0` to `23`. In this example, `hour = 1` means the observation occurred between 1:00 AM and 1:59 AM.
- **`day_of_week`** represents the day of the week using values from `0` to `6`, where `0` is Monday and `6` is Sunday. In this example, `day_of_week = 2` represents Wednesday.

Since the first observations occur on **Wednesday, February 1, 2017 at 1:02 AM**, the extracted values are `hour = 1` and `day_of_week = 2`.

These features can help capture recurring time-related operating patterns while the original timestamp remains available for maintaining the chronological order of observations.

### Creating Lag Features

Lag features represent the value of a sensor at previous timestamps.

For a time-ordered dataset, a lag of 1 contains the sensor value from the immediately preceding observation. A lag of 5 contains the value from five observations earlier.

These features allow the model to use recent historical sensor information when making a prediction at the current timestamp.

Lag features are created separately within each file/instance so that observations from different files are never treated as part of the same continuous timeline.

In [9]:
# Create lag features for the sensor columns in the first processed file.

lag_periods = [1, 5]

sample_data = pd.read_parquet(processed_files[0])


# Create the lagged sensor values.

for sensor in sensor_columns:
    for lag in lag_periods:
        feature_name = f"{sensor}_lag_{lag}"
        sample_data[feature_name] = sample_data[sensor].shift(lag)


# Display the original and lagged values for one sensor.

example_sensor = "P-TPT"

columns_to_display = [
    example_sensor,
    f"{example_sensor}_lag_1",
    f"{example_sensor}_lag_5"
]

print(sample_data[columns_to_display].head(8))

                          P-TPT  P-TPT_lag_1  P-TPT_lag_5
timestamp                                                
2017-02-01 01:02:07  10074540.0          NaN          NaN
2017-02-01 01:02:08  10074540.0   10074540.0          NaN
2017-02-01 01:02:09  10074540.0   10074540.0          NaN
2017-02-01 01:02:10  10074540.0   10074540.0          NaN
2017-02-01 01:02:11  10074540.0   10074540.0          NaN
2017-02-01 01:02:12  10074540.0   10074540.0   10074540.0
2017-02-01 01:02:13  10074540.0   10074540.0   10074540.0
2017-02-01 01:02:14  10074540.0   10074540.0   10074540.0


### Creating Rolling-Window Features

Rolling features summarize sensor behavior over a recent time window.

For each sensor, rolling mean and rolling standard deviation are calculated using previous observations. These features capture both the recent level of a sensor and the amount of short-term variation.

A window of 60 observations is used, which corresponds to approximately one minute because the 3W dataset is sampled at one-second intervals.

The rolling calculations use only current and previous observations, ensuring that future sensor values are not included.

In [8]:
# Define the rolling-window size.

rolling_window = 60


# Create rolling features for the sensor columns in the sample file.

for sensor in sensor_columns:
    mean_feature = f"{sensor}_rolling_mean_{rolling_window}"
    std_feature = f"{sensor}_rolling_std_{rolling_window}"

    sample_data[mean_feature] = (
        sample_data[sensor]
        .rolling(
            window=rolling_window,
            min_periods=1
        )
        .mean()
    )

    sample_data[std_feature] = (
        sample_data[sensor]
        .rolling(
            window=rolling_window,
            min_periods=1
        )
        .std()
    )


# Display the rolling features for one sensor.

example_sensor = "P-TPT"

columns_to_display = [
    example_sensor,
    f"{example_sensor}_rolling_mean_{rolling_window}",
    f"{example_sensor}_rolling_std_{rolling_window}"
]

print(sample_data[columns_to_display].head(8))

                          P-TPT  P-TPT_rolling_mean_60  P-TPT_rolling_std_60
timestamp                                                                   
2017-02-01 01:02:07  10074540.0             10074540.0                   NaN
2017-02-01 01:02:08  10074540.0             10074540.0                   0.0
2017-02-01 01:02:09  10074540.0             10074540.0                   0.0
2017-02-01 01:02:10  10074540.0             10074540.0                   0.0
2017-02-01 01:02:11  10074540.0             10074540.0                   0.0
2017-02-01 01:02:12  10074540.0             10074540.0                   0.0
2017-02-01 01:02:13  10074540.0             10074540.0                   0.0
2017-02-01 01:02:14  10074540.0             10074540.0                   0.0


### Creating Rate-of-Change Features

Rate-of-change features measure how much a sensor value changes between the current observation and a previous observation.

For each sensor, the difference from the immediately preceding observation is calculated. This helps capture sudden increases or decreases that may be more informative for event detection than the absolute sensor value alone.

The calculation uses the current value and the previous observation within the same file/instance, so information from future observations is not used.

In [10]:
# Create rate-of-change features for the sensor columns.

for sensor in sensor_columns:
    rate_feature = f"{sensor}_rate_of_change"

    sample_data[rate_feature] = (
        sample_data[sensor]
        .diff()
    )


# Display the rate of change for one sensor.

example_sensor = "P-TPT"

columns_to_display = [
    example_sensor,
    f"{example_sensor}_rate_of_change"
]

print(sample_data[columns_to_display].head(8))

                          P-TPT  P-TPT_rate_of_change
timestamp                                            
2017-02-01 01:02:07  10074540.0                   NaN
2017-02-01 01:02:08  10074540.0                   0.0
2017-02-01 01:02:09  10074540.0                   0.0
2017-02-01 01:02:10  10074540.0                   0.0
2017-02-01 01:02:11  10074540.0                   0.0
2017-02-01 01:02:12  10074540.0                   0.0
2017-02-01 01:02:13  10074540.0                   0.0
2017-02-01 01:02:14  10074540.0                   0.0


### Creating Trend and Variability Features

Short-term sensor behavior can contain useful information beyond the current value and simple rate of change.

For each continuous sensor, the difference between the current value and its recent rolling mean is calculated. This feature indicates whether the current sensor value is above or below its recent local level.

A rolling standard deviation is already available from the previous step and represents short-term sensor variability.

These features help describe both the direction of the current value relative to its recent behavior and the stability of the sensor over the selected window.

In [12]:
# Recreate the rolling features required for the deviation calculation.

for sensor in sensor_columns:
    mean_feature = f"{sensor}_rolling_mean_{rolling_window}"
    std_feature = f"{sensor}_rolling_std_{rolling_window}"

    sample_data[mean_feature] = (
        sample_data[sensor]
        .rolling(
            window=rolling_window,
            min_periods=1
        )
        .mean()
    )

    sample_data[std_feature] = (
        sample_data[sensor]
        .rolling(
            window=rolling_window,
            min_periods=1
        )
        .std()
    )


# Create the deviation-from-rolling-mean feature.

for sensor in sensor_columns:
    rolling_mean_feature = f"{sensor}_rolling_mean_{rolling_window}"
    deviation_feature = f"{sensor}_deviation_from_mean_{rolling_window}"

    sample_data[deviation_feature] = (
        sample_data[sensor]
        - sample_data[rolling_mean_feature]
    )


# Display the trend and variability features for one sensor.

example_sensor = "P-TPT"

columns_to_display = [
    example_sensor,
    f"{example_sensor}_rolling_mean_{rolling_window}",
    f"{example_sensor}_deviation_from_mean_{rolling_window}",
    f"{example_sensor}_rolling_std_{rolling_window}"
]

print(sample_data[columns_to_display].head(8))

                          P-TPT  P-TPT_rolling_mean_60  \
timestamp                                                
2017-02-01 01:02:07  10074540.0             10074540.0   
2017-02-01 01:02:08  10074540.0             10074540.0   
2017-02-01 01:02:09  10074540.0             10074540.0   
2017-02-01 01:02:10  10074540.0             10074540.0   
2017-02-01 01:02:11  10074540.0             10074540.0   
2017-02-01 01:02:12  10074540.0             10074540.0   
2017-02-01 01:02:13  10074540.0             10074540.0   
2017-02-01 01:02:14  10074540.0             10074540.0   

                     P-TPT_deviation_from_mean_60  P-TPT_rolling_std_60  
timestamp                                                                
2017-02-01 01:02:07                           0.0                   NaN  
2017-02-01 01:02:08                           0.0                   0.0  
2017-02-01 01:02:09                           0.0                   0.0  
2017-02-01 01:02:10                           0.0

### Handling Missing Values Created by Feature Engineering

Lag and rolling features naturally introduce missing values at the beginning of each file/instance. For example, a lag of 5 cannot be calculated for the first five observations because earlier observations do not exist.

These missing values are structural consequences of feature creation rather than missing sensor measurements.

They will be retained at this stage instead of being filled with arbitrary values. The affected initial observations can be removed later after the complete feature set and prediction target have been constructed.

In [13]:
# Count missing values created by the temporal features.

temporal_feature_columns = [
    column
    for column in sample_data.columns
    if (
        "_lag_" in column
        or "_rolling_" in column
        or "_rate_of_change" in column
        or "_deviation_from_mean_" in column
    )
]


# Calculate the number of missing values in these features.

temporal_missing = (
    sample_data[temporal_feature_columns]
    .isna()
    .sum()
)


# Display only features that contain missing values.

print(
    temporal_missing[
        temporal_missing > 0
    ]
)

P-TPT_lag_1                             1
P-TPT_lag_5                             5
P-MON-CKP_lag_1                         1
P-MON-CKP_lag_5                         5
P-PDG_lag_1                             1
P-PDG_lag_5                             5
T-TPT_lag_1                             1
T-TPT_lag_5                             5
T-JUS-CKP_lag_1                         1
T-JUS-CKP_lag_5                         5
P-JUS-CKGL_lag_1                        1
P-JUS-CKGL_lag_5                        5
P-ANULAR_lag_1                          1
P-ANULAR_lag_5                          5
QGL_lag_1                               1
QGL_lag_5                               5
ESTADO-SDV-P_lag_1                      1
ESTADO-SDV-P_lag_5                      5
T-MON-CKP_lag_1                     21474
T-MON-CKP_lag_5                     21474
P-JUS-CKP_lag_1                     21474
P-JUS-CKP_lag_5                     21474
T-PDG_lag_1                             1
T-PDG_lag_5                       

### Understanding Missing Values From Feature Engineering

The output shows missing values introduced by the temporal feature calculations.

- **Lag features:** `lag_1` produces one missing value and `lag_5` produces five missing values at the beginning of the file/instance because the required previous observations do not exist.
- **Rate-of-change features:** the first observation has no previous value, so it produces one missing value.
- **Rolling standard deviation:** the first observation has only one value available, so its standard deviation cannot be calculated.
- **Sensors that were already completely missing in this file/instance**, such as `T-MON-CKP` and `P-JUS-CKP`, remain missing in their derived features. Feature engineering does not create information where the original sensor value is unavailable.

Therefore, the missing values shown here are expected. The initial temporal-feature gaps are structural, while the larger missing blocks reflect the original sensor availability in the file/instance. No arbitrary values are introduced to fill them at this stage.

### Inspecting Feature Redundancy

Feature engineering creates multiple representations of the same underlying sensor behavior, such as the original value, lagged values, rolling statistics, and rate of change.

Before building the final feature dataset, the engineered features are inspected for excessive redundancy. Highly correlated features may provide very similar information and can increase model complexity without adding meaningful predictive information.

This step is used as an exploratory check. Features will not be removed solely because they are correlated; their usefulness will be considered together with the model and evaluation results.

In [14]:
# Select numeric features currently available in the sample data.

numeric_features = sample_data.select_dtypes(
    include="number"
).columns.tolist()


# Calculate the absolute correlation between numeric features.

correlation_matrix = (
    sample_data[numeric_features]
    .corr()
    .abs()
)


# Find feature pairs with very high correlation.

correlation_threshold = 0.95

high_correlation_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        feature_1 = correlation_matrix.columns[i]
        feature_2 = correlation_matrix.columns[j]
        correlation = correlation_matrix.iloc[i, j]

        if correlation >= correlation_threshold:
            high_correlation_pairs.append(
                {
                    "feature_1": feature_1,
                    "feature_2": feature_2,
                    "absolute_correlation": correlation
                }
            )


high_correlation_df = pd.DataFrame(
    high_correlation_pairs
).sort_values(
    "absolute_correlation",
    ascending=False
)


print(
    "Highly correlated feature pairs:",
    len(high_correlation_df)
)

print(
    high_correlation_df.head(20)
)

Highly correlated feature pairs: 28
           feature_1                   feature_2  absolute_correlation
10        P-JUS-CKGL            P-JUS-CKGL_lag_5              1.000000
24  P-JUS-CKGL_lag_5  P-JUS-CKGL_rolling_mean_60              1.000000
23  P-JUS-CKGL_lag_1  P-JUS-CKGL_rolling_mean_60              1.000000
11        P-JUS-CKGL  P-JUS-CKGL_rolling_mean_60              1.000000
22  P-JUS-CKGL_lag_1            P-JUS-CKGL_lag_5              1.000000
9         P-JUS-CKGL            P-JUS-CKGL_lag_1              1.000000
12          P-ANULAR              P-ANULAR_lag_1              0.999997
0              P-TPT                 P-TPT_lag_1              0.999989
4              T-TPT                 T-TPT_lag_1              0.999983
25    P-ANULAR_lag_1              P-ANULAR_lag_5              0.999959
13          P-ANULAR              P-ANULAR_lag_5              0.999937
15       P-TPT_lag_1                 P-TPT_lag_5              0.999834
1              P-TPT                 P-TP

### Understanding Feature Correlation

The correlation check identified **28 feature pairs with an absolute correlation of at least 0.95**.

Several lagged and rolling features have very high correlation with their original sensor values. This is expected because sensors such as `P-TPT`, `P-ANULAR`, and `T-TPT` change relatively slowly over time, so their recent values remain similar to the current value.

For example:

- `P-TPT` and `P-TPT_lag_1` have a correlation of approximately **0.99999**.
- `P-TPT` and `P-TPT_lag_5` have a correlation of approximately **0.99975**.
- `P-JUS-CKGL` and its lagged/rolling features have correlations of **1.00** in this sample.

High correlation does not automatically mean that a feature should be removed. Lag and rolling features can still provide useful temporal information for event detection even when they are strongly correlated with the original sensor.

Therefore, this analysis is treated as a **diagnostic check rather than an automatic feature-removal rule**. Feature selection will be considered after the complete feature set and modeling approach are established.

### Checking for Temporal Leakage

Temporal features must not contain information from future observations.

The lag, rate-of-change, and rolling features created in this notebook use the current observation and previously observed values. However, the feature-engineering process must also preserve the chronological order within every file/instance.

A direct comparison between the original timestamps and the feature-engineered data is therefore used to confirm that the timestamp ordering has not changed.

This check helps ensure that the temporal structure required for trustworthy event prediction is preserved before the final feature dataset is created.

In [15]:
# Check that timestamps remain sorted within the sample file.

timestamps_are_sorted = sample_data.index.is_monotonic_increasing


# Check for duplicate timestamps.

duplicate_timestamps = sample_data.index.duplicated().sum()


# Display the temporal-order checks.

print("Timestamps sorted:", timestamps_are_sorted)
print("Duplicate timestamps:", duplicate_timestamps)

Timestamps sorted: True
Duplicate timestamps: 0


### Building the Feature-Engineered Dataset

The feature-engineering process is now applied to all 2,228 processed files/instances.

For each file/instance, timestamp, sensor, lag, rolling, rate-of-change, and deviation features are generated while preserving the original file boundaries and chronological order.

The resulting feature-engineered files are stored separately from the processed dataset. The original processed data remains unchanged.

The final validation confirms that the number of files and observation counts are preserved and that timestamps remain ordered.

In [18]:
# Define the output directory for the final feature-engineered dataset.

feature_engineered_path = (
    project_path
    / "data"
    / "processed"
    / "3W_feature_engineered"
)


# Remove any incomplete output from the previous attempt.

import shutil

if feature_engineered_path.exists():
    shutil.rmtree(feature_engineered_path)

feature_engineered_path.mkdir(
    parents=True,
    exist_ok=True
)


# Define the temporal settings used for feature creation.

lag_periods = [1, 5]
rolling_window = 60


# Define the columns that will be retained in the final dataset.

final_label_columns = [
    "class",
    "state"
]

final_time_columns = [
    "hour",
    "day_of_week"
]


# Process each file/instance one at a time.

processed_count = 0

for file_path in processed_files:

    data = pd.read_parquet(file_path)

    # Create timestamp-based features.

    data["hour"] = data.index.hour
    data["day_of_week"] = data.index.dayofweek

    # Store engineered features separately.

    engineered_features = pd.DataFrame(
        index=data.index
    )

    # Create lag features.

    for sensor in sensor_columns:
        for lag in lag_periods:

            feature_name = f"{sensor}_lag_{lag}"

            engineered_features[feature_name] = (
                data[sensor].shift(lag)
            )

    # Create rolling features.

    for sensor in sensor_columns:

        mean_feature = (
            f"{sensor}_rolling_mean_{rolling_window}"
        )

        std_feature = (
            f"{sensor}_rolling_std_{rolling_window}"
        )

        engineered_features[mean_feature] = (
            data[sensor]
            .rolling(
                window=rolling_window,
                min_periods=1
            )
            .mean()
        )

        engineered_features[std_feature] = (
            data[sensor]
            .rolling(
                window=rolling_window,
                min_periods=1
            )
            .std()
        )

    # Create rate-of-change and deviation features.

    for sensor in sensor_columns:

        rate_feature = (
            f"{sensor}_rate_of_change"
        )

        mean_feature = (
            f"{sensor}_rolling_mean_{rolling_window}"
        )

        deviation_feature = (
            f"{sensor}_deviation_from_mean_{rolling_window}"
        )

        engineered_features[rate_feature] = (
            data[sensor].diff()
        )

        engineered_features[deviation_feature] = (
            data[sensor]
            - engineered_features[mean_feature]
        )

    # Add timestamp features and labels.

    engineered_features["hour"] = data["hour"]
    engineered_features["day_of_week"] = data["day_of_week"]

    engineered_features["class"] = data["class"]
    engineered_features["state"] = data["state"]

    # Save the compact feature-engineered file.

    output_folder = (
        feature_engineered_path
        / file_path.parent.name
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = (
        output_folder
        / file_path.name
    )

    engineered_features.to_parquet(
        output_file
    )

    processed_count += 1

    del data
    del engineered_features


# Validate the number of generated files.

feature_engineered_files = sorted(
    feature_engineered_path.glob("*/*.parquet")
)

print("Feature engineering completed.")
print("--------------------------------")
print("Input files:", len(processed_files))
print("Processed files:", processed_count)
print("Output files:", len(feature_engineered_files))

OSError: [Errno 28] Error writing bytes to file. Detail: [errno 28] No space left on device